## BIBLIOTECAS

### INSTALAÇÃO

In [9]:
!pip install scikit_learn
!pip install pandas


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### LIBS

In [10]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import accuracy_score, classification_report
from sklearn.linear_model import LogisticRegression


## MACHINE LEARNING

### SEPARAÇÃO DOS DATAFRAMES

In [11]:
# Dataframe de testes
df_phrase = pd.read_csv(rf"C:\Users\davih\OneDrive\Documentos\PROJETOS\Fiap\ANO 2\Cardio IA\config\Frases_simuladas.txt", header=None, names=['frases'])
display(df_phrase)

,frases
0,Cerca de uma semana sinto um formigamento cons...
1,Ha tres dias sinto um zumbido persistente no o...
2,Sinto um aperto na garganta e uma leve falta d...
3,Apresento uma tosse seca e persistente ha duas...
4,Estou com uma sensibilidade extrema nos dentes...
5,Ha dois dias sinto dor nas costas ao ficar mui...
6,Desde a semana passada sinto um cansaco excess...
7,Desde ontem sinto uma pressao na regiao dos ol...
8,"Estou com dor no pescoco por tres dias, especi..."
9,Desde a semana passada tenho notado perda de a...


In [12]:
# Carregar a base que você criou
df = pd.read_csv(r"C:\Users\davih\OneDrive\Documentos\PROJETOS\Fiap\ANO 2\Cardio IA\config\base_treinamento_triagem.csv")
display(df)

,frase,situacao
0,Sinto uma dor forte no peito e falta de ar,alto risco
1,Estou com muita dificuldade para respirar e do...,alto risco
2,Meu coração está batendo muito rápido e sinto ...,alto risco
3,Dor aguda no tórax que irradia para as costas,alto risco
4,Sinto um aperto no peito como se fosse um peso,alto risco
5,Falta de ar extrema mesmo em repouso,alto risco
6,Cansaço excessivo e inchaço nas pernas,alto risco
7,Desmaiei após sentir palpitações fortes,alto risco
8,Suor frio e dor intensa no peito,alto risco
9,Não consigo respirar direito faz horas,alto risco


### VETORIZAÇÃO

In [13]:
# Vetorização TF-IDF
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df['frase'])
y = df['situacao']

X_phrase = vectorizer.transform(df_phrase['frases'])

In [14]:

# Divisão Treino/Teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


### TREINAMENTO DE MODELOS

#### DECISION TREE

In [15]:
# Treinamento com Decision Tree
modelo_dt = DecisionTreeClassifier(criterion='entropy', random_state=42)
modelo_dt.fit(X_train, y_train)

# Avaliação do Desempenho
y_pred = modelo_dt.predict(X_test)
print(f"Acurácia: {accuracy_score(y_test, y_pred):.2f}")
print(classification_report(y_test, y_pred))

# Visualizar as regras lógicas criadas pela IA
regras = export_text(modelo_dt, feature_names=list(vectorizer.get_feature_names_out()))
print("\nRegras da Árvore de Decisão:\n", regras)

Acurácia: 0.29
              precision    recall  f1-score   support

  alto risco       0.00      0.00      0.00         4
 baixo risco       0.33      0.67      0.44         3

    accuracy                           0.29         7
   macro avg       0.17      0.33      0.22         7
weighted avg       0.14      0.29      0.19         7


Regras da Árvore de Decisão:
 |--- leve <= 0.14
|   |--- sinto <= 0.13
|   |   |--- um <= 0.13
|   |   |   |--- cansaço <= 0.19
|   |   |   |   |--- para <= 0.17
|   |   |   |   |   |--- ar <= 0.19
|   |   |   |   |   |   |--- persistente <= 0.29
|   |   |   |   |   |   |   |--- fortes <= 0.24
|   |   |   |   |   |   |   |   |--- class: baixo risco
|   |   |   |   |   |   |   |--- fortes >  0.24
|   |   |   |   |   |   |   |   |--- class: alto risco
|   |   |   |   |   |   |--- persistente >  0.29
|   |   |   |   |   |   |   |--- class: alto risco
|   |   |   |   |   |--- ar >  0.19
|   |   |   |   |   |   |--- class: alto risco
|   |   |   |   |---

#### LOGISTC REGRESSION

In [16]:
# Treinamento com Logistic Regression
modelo_lr = LogisticRegression()
modelo_lr.fit(X_train, y_train)

# Avaliação do Desempenho
y_pred = modelo_lr.predict(X_test)
print(f"Acurácia: {accuracy_score(y_test, y_pred):.2f}")
print(classification_report(y_test, y_pred))

# Testar com frases novas
novas_frases = ["Sinto um aperto na garganta e uma falta de ar", "Estou com sensibilidade nos dentes"]
X_new = vectorizer.transform(novas_frases)
previsoes = modelo_lr.predict(X_new)

for frase, risco in zip(novas_frases, previsoes):
    print(f"Frase: {frase} -> Classificação: {risco}")

Acurácia: 0.57
              precision    recall  f1-score   support

  alto risco       1.00      0.25      0.40         4
 baixo risco       0.50      1.00      0.67         3

    accuracy                           0.57         7
   macro avg       0.75      0.62      0.53         7
weighted avg       0.79      0.57      0.51         7

Frase: Sinto um aperto na garganta e uma falta de ar -> Classificação: baixo risco
Frase: Estou com sensibilidade nos dentes -> Classificação: baixo risco


### TESTES

#### DECISION TREE

In [17]:
df_decision_tree = df_phrase
df_decision_tree['situacao'] = modelo_dt.predict(X_phrase)
display(df_phrase)


,frases,situacao
0,Cerca de uma semana sinto um formigamento cons...,alto risco
1,Ha tres dias sinto um zumbido persistente no o...,alto risco
2,Sinto um aperto na garganta e uma leve falta d...,baixo risco
3,Apresento uma tosse seca e persistente ha duas...,alto risco
4,Estou com uma sensibilidade extrema nos dentes...,alto risco
5,Ha dois dias sinto dor nas costas ao ficar mui...,alto risco
6,Desde a semana passada sinto um cansaco excess...,alto risco
7,Desde ontem sinto uma pressao na regiao dos ol...,alto risco
8,"Estou com dor no pescoco por tres dias, especi...",baixo risco
9,Desde a semana passada tenho notado perda de a...,baixo risco


#### LOGISTIC REGRESSION

In [18]:
df_logistic_regression = df_phrase
df_logistic_regression['situacao'] = modelo_lr.predict(X_phrase)
display(df_phrase)

,frases,situacao
0,Cerca de uma semana sinto um formigamento cons...,baixo risco
1,Ha tres dias sinto um zumbido persistente no o...,baixo risco
2,Sinto um aperto na garganta e uma leve falta d...,baixo risco
3,Apresento uma tosse seca e persistente ha duas...,baixo risco
4,Estou com uma sensibilidade extrema nos dentes...,baixo risco
5,Ha dois dias sinto dor nas costas ao ficar mui...,baixo risco
6,Desde a semana passada sinto um cansaco excess...,baixo risco
7,Desde ontem sinto uma pressao na regiao dos ol...,baixo risco
8,"Estou com dor no pescoco por tres dias, especi...",baixo risco
9,Desde a semana passada tenho notado perda de a...,baixo risco
